In [ ]:
!pip install --upgrade torch torchvision
!pip install timm --no-deps

In [ ]:
import timm
import torch
import torch.nn as nn

# Verification
print(f"PyTorch version: {torch.__version__}")
try:
    test_gelu = nn.GELU()
    print("✅ nn.GELU is present.")
except AttributeError:
    print("❌ GELU error still persists.")

In [ ]:
import cv2
import os
import json
import pandas as pd
from tqdm import tqdm

# Define Paths
BASE_DIR = '/kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos/'
OUTPUT_REAL = '/kaggle/working/extracted_images/real'
OUTPUT_FAKE = '/kaggle/working/extracted_images/fake'

# Create directories
os.makedirs(OUTPUT_REAL, exist_ok=True)
os.makedirs(OUTPUT_FAKE, exist_ok=True)

# Load Metadata
with open(os.path.join(BASE_DIR, 'metadata.json')) as f:
    metadata = json.load(f)

# Extraction Loop
print("🚀 Extracting frames based on metadata labels...")

# We loop through the metadata dictionary
for video_file, info in tqdm(metadata.items()):
    label = info['label'] # 'REAL' or 'FAKE'
    video_path = os.path.join(BASE_DIR, video_file)
    
    # Target output folder
    target_folder = OUTPUT_REAL if label == 'REAL' else OUTPUT_FAKE
    output_path = os.path.join(target_folder, f"{video_file.split('.')[0]}.jpg")
    
    # Extract the first frame
    cap = cv2.VideoCapture(video_path)
    success, frame = cap.read()
    
    if success:
        # Resize to 512x512 t
        frame = cv2.resize(frame, (512, 512))
        cv2.imwrite(output_path, frame)
    
    cap.release()

# Count Check
print(f"✅ Extraction Complete.")
print(f"Real images: {len(os.listdir(OUTPUT_REAL))}")
print(f"Fake images: {len(os.listdir(OUTPUT_FAKE))}")

In [5]:
# import shutil

# # This will create 'extracted_images.zip' in /kaggle/working/
# shutil.make_archive('extracted_images', 'zip', '/kaggle/working/extracted_images')

# print("✅ Archive created successfully: extracted_images.zip")

In [6]:
import os
import timm
import torch
import pandas as pd
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

In [ ]:
dinov2_models = timm.list_models('*dino*')
print(dinov2_models)

In [8]:
# --- DATASET CLASS ---
class SFTDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        
        # Mapping: Real = 0, Fake = 1
        for label, folder in enumerate(['real', 'fake']):
            folder_path = os.path.join(root_dir, folder)
            if not os.path.exists(folder_path):
                continue
            for f in os.listdir(folder_path):
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((os.path.join(folder_path, f), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)

In [9]:
# --- MODEL ARCHITECTURE ---
class DeepfakeSentinel(nn.Module):
    def __init__(self):
        super().__init__()
        # Load DINOv2 Small (SSL backbone)
        self.backbone = timm.create_model('vit_small_patch16_224_dino', pretrained=True, num_classes=0)
        # Freeze backbone
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        self.head = nn.Sequential(
            nn.Linear(384, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1) # Binary output (Logit)
        )

    def forward(self, x):
        # Backbone is frozen, so no gradients needed here
        with torch.no_grad():
            features = self.backbone(x)
        return self.head(features)

In [ ]:
# --- INITIALIZATION ---
data_dir = '/kaggle/working/extracted_images/' 
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = SFTDataset(data_dir, transform=transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = DeepfakeSentinel().cuda()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-4)

In [11]:
# --- SFT TRAINING LOOP ---
scaler = GradScaler()
epochs = 10 # Increased epochs because 400 images is a small dataset
print(f"🚀 Starting SFT on {len(dataset)} images...")

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for imgs, labels in loader:
        imgs, labels = imgs.cuda(), labels.cuda().unsqueeze(1)
        
        optimizer.zero_grad()
        with autocast(): 
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward() 
        scaler.step(optimizer)
        scaler.update()
        # loss.backward()
        # optimizer.step()
        
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(loader):.4f}")

# Save the SFT baseline - this is our "Reference Model" for DPO
torch.save(model.state_dict(), 'sft_model.pth')
print("✅ SFT Phase Complete. 'sft_model.pth' saved.")

🚀 Starting SFT on 400 images...
Epoch 1/10 | Loss: 0.7985
Epoch 2/10 | Loss: 0.4969
Epoch 3/10 | Loss: 0.4946
Epoch 4/10 | Loss: 0.4790
Epoch 5/10 | Loss: 0.4507
Epoch 6/10 | Loss: 0.4440
Epoch 7/10 | Loss: 0.4375
Epoch 8/10 | Loss: 0.4415
Epoch 9/10 | Loss: 0.4395
Epoch 10/10 | Loss: 0.4288
✅ SFT Phase Complete. 'sft_model.pth' saved.


In [16]:
# --- DPO PAIR DATASET ---
class DPODataset(Dataset):
    def __init__(self, csv_path, fake_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.fake_dir = fake_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # winner = more convincing/dangerous, loser = obvious fake
        win_path = os.path.join(self.fake_dir, row['winner'])
        lose_path = os.path.join(self.fake_dir, row['loser'])
        
        win_img = Image.open(win_path).convert('RGB')
        lose_img = Image.open(lose_path).convert('RGB')
        
        if self.transform:
            win_img = self.transform(win_img)
            lose_img = self.transform(lose_img)
            
        return win_img, lose_img

def dpo_loss_with_margin(policy_logits, ref_logits, beta=0.5, margin=0.1):
    policy_logps = F.logsigmoid(policy_logits)
    ref_logps = F.logsigmoid(ref_logits)
    
    pi_logratios = policy_logps[:, 0] - policy_logps[:, 1]
    ref_logratios = ref_logps[:, 0] - ref_logps[:, 1]
    
    # We add a margin to force the model to 'clearly' prefer the winner
    logits = pi_logratios - ref_logratios - margin
    loss = -F.logsigmoid(beta * logits).mean()
    return loss

# --- THE DPO LOSS FUNCTION ---
def dpo_loss(policy_logits, ref_logits, beta=0.5):
    """
    beta: controls the KL penalty. Higher beta = stay closer to SFT model.
    """
    # policy_logits: [batch, 2] -> [win_logit, lose_logit]
    # ref_logits: [batch, 2] -> [win_logit, lose_logit]
    
    # We use logsigmoid to get log-probabilities of "fakeness"
    policy_logps = F.logsigmoid(policy_logits)
    ref_logps = F.logsigmoid(ref_logits)
    
    # Calculate log ratios
    # win is at index 0, lose is at index 1
    pi_logratios = policy_logps[:, 0] - policy_logps[:, 1]
    ref_logratios = ref_logps[:, 0] - ref_logps[:, 1]
    
    # DPO objective: maximize the difference between pi and ref log-ratios
    losses = -F.logsigmoid(beta * (pi_logratios - ref_logratios))
    return losses.mean()

In [23]:
# --- INITIALIZE MODELS ---
# Use the same class definition from your SFT cell
ref_model = DeepfakeSentinel().cuda()
ref_model.load_state_dict(torch.load('sft_model.pth'))
ref_model.eval() 

# Initialize Policy Model and UNFREEZE the last layer
policy_model = DeepfakeSentinel().cuda()
policy_model.load_state_dict(torch.load('sft_model.pth'))

# Unfreeze the last block of DINOv2 (Block 11)
# This allows the model to learn subtle 'Deepfake-specific' features
for name, param in policy_model.backbone.named_parameters():
    if "blocks.11" in name or "norm" in name:
        param.requires_grad = True

# Optimized Optimizer
# We use a higher LR for the head and a lower one for the backbone
optimizer = torch.optim.AdamW([
    {'params': policy_model.head.parameters(), 'lr': 5e-5},
    {'params': policy_model.backbone.parameters(), 'lr': 1e-6} # Very slow for backbone
], weight_decay=0.01)

# --- EXECUTION ---
csv_path = '/kaggle/input/datasets/torshamajumder/deepfake/preferences.csv' # Path to your uploaded file
fake_dir = '/kaggle/working/extracted_images/fake'

dpo_dataset = DPODataset(csv_path, fake_dir, transform=transform)
dpo_loader = DataLoader(dpo_dataset, batch_size=16, shuffle=True)

print(f"🛡️ Aligning with {len(dpo_dataset)} Human Preferences...")

epochs = 10 
for epoch in range(epochs):
    policy_model.train()
    epoch_loss = 0
    
    for win_imgs, lose_imgs in dpo_loader:
        win_imgs, lose_imgs = win_imgs.cuda(), lose_imgs.cuda()
        
        # Combine win/lose into one batch for efficiency
        batch = torch.cat([win_imgs, lose_imgs], dim=0)
        
        # Get Reference Logits (No gradients)
        with torch.no_grad():
            ref_all = ref_model(batch)
            ref_win, ref_lose = ref_all[:len(win_imgs)], ref_all[len(win_imgs):]
            
        # Get Policy Logits
        policy_all = policy_model(batch)
        policy_win, policy_lose = policy_all[:len(win_imgs)], policy_all[len(win_imgs):]
        
        # Combine into [batch, 2] format for the loss function
        policy_pair = torch.cat([policy_win, policy_lose], dim=1)
        ref_pair = torch.cat([ref_win, ref_lose], dim=1)
        
        loss = dpo_loss_with_margin(policy_pair, ref_pair)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    print(f"DPO Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(dpo_loader):.4f}")

torch.save(policy_model.state_dict(), 'dpo_aligned_model.pth')
print("✅ DPO Aligned Model saved.")

🛡️ Aligning with 106 Human Preferences...
DPO Epoch 1/10 | Loss: 0.7232
DPO Epoch 2/10 | Loss: 0.7015
DPO Epoch 3/10 | Loss: 0.7022
DPO Epoch 4/10 | Loss: 0.6963
DPO Epoch 5/10 | Loss: 0.7003
DPO Epoch 6/10 | Loss: 0.6918
DPO Epoch 7/10 | Loss: 0.6768
DPO Epoch 8/10 | Loss: 0.6774
DPO Epoch 9/10 | Loss: 0.6696
DPO Epoch 10/10 | Loss: 0.6643
✅ DPO Aligned Model saved.


In [25]:
policy_model.eval()
ref_model.eval()

# Pick one of your 135 pairs
win_img, lose_img = dpo_dataset[0]
win_img, lose_img = win_img.unsqueeze(0).cuda(), lose_img.unsqueeze(0).cuda()

with torch.no_grad():
    # Check "Suspicion Score" (Logits)
    ref_scores = ref_model(torch.cat([win_img, lose_img]))
    pol_scores = policy_model(torch.cat([win_img, lose_img]))

print(f"REF MODEL (SFT) - Win Logit: {ref_scores[0].item():.4f}, Lose Logit: {ref_scores[1].item():.4f}")
print(f"POL MODEL (DPO) - Win Logit: {pol_scores[0].item():.4f}, Lose Logit: {pol_scores[1].item():.4f}")

REF MODEL (SFT) - Win Logit: 1.7415, Lose Logit: 2.2002
POL MODEL (DPO) - Win Logit: -0.0030, Lose Logit: 2.2971
